# Intro to Numba

Numba is an open source, NumPy-aware optimizing compiler for Python. It uses the LLVM compiler project to generate machine code from Python syntax.

With this intro, lets break down the above sentence and understand what it means. 

- Python is easy to write but slow for number-crunching loops.
- C is fast but is syntax strict, and difficult to iterate on quick. 
- **Numba** gives you C like speed with Python syntax.

We'll time the same algorithm three ways- Python, C and (Python + Numba) .

## Sum of Squares

Compute $\sum x_i^2$ over an array of one million numbers- loop over data.

### Python version

In [297]:
import time
import numpy as np

def sum_of_squares(arr):
    s = 0.0
    for x in arr:
        s += x * x
    return s

arr = np.arange(1, 1_000_001)
print(f'len(arr) = {len(arr)}')
print(f'arr[0], arr[1], arr[2], arr[3], ... arr[-1] = {arr[0]}, {arr[1]}, {arr[2]}, {arr[3]}, ... {arr[-1]}')


t0 = time.perf_counter()
result = sum_of_squares(arr)
py_sum_time = time.perf_counter() - t0

print(f"result = {result:.0f}")
print(f"time   = {py_sum_time * 1000:.1f} ms")

len(arr) = 1000000
arr[0], arr[1], arr[2], arr[3], ... arr[-1] = 1, 2, 3, 4, ... 1000000
result = 333333833333127552
time   = 94.3 ms


### C version

Same loop. But we need to: write a file, choose types, allocate memory, compile, then run.

In [298]:
%%writefile sum_sq.c
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

int main(void) {
    int n = 1000000;
    double *arr = malloc(n * sizeof(double));
    for (int i = 0; i < n; i++) arr[i] = i + 1;

    struct timespec t0, t1;
    clock_gettime(CLOCK_MONOTONIC, &t0);
    double s = 0.0;
    for (int i = 0; i < n; i++) s += arr[i] * arr[i];
    clock_gettime(CLOCK_MONOTONIC, &t1);

    double elapsed = (t1.tv_sec - t0.tv_sec) + (t1.tv_nsec - t0.tv_nsec) / 1e9;
    printf("result = %.0f\ntime   = %.3f ms\n", s, elapsed * 1000);
    free(arr);
}

Overwriting sum_sq.c


In [299]:
import shutil
CC = shutil.which('clang') or shutil.which('gcc')
CC = shutil.which('gcc')
print(f'using: {CC}')

using: /usr/bin/gcc


In [300]:
!{CC} -O1 -o sum_sq_O1 sum_sq.c && echo "--- {CC} -O1 ---" && ./sum_sq_O1

--- /usr/bin/gcc -O1 ---
result = 333333833333127552
time   = 1.554 ms


In [301]:
!{CC} -O2 -o sum_sq_O2 sum_sq.c && echo "--- {CC} -O2 ---" && ./sum_sq_O2
!{CC} -O3 -o sum_sq_O3 sum_sq.c && echo "--- {CC} -O3 ---" && ./sum_sq_O3

--- /usr/bin/gcc -O2 ---
result = 333333833333127552
time   = 0.681 ms
--- /usr/bin/gcc -O3 ---
result = 333333833333127552
time   = 0.621 ms


### with Numba

Same Python function. To use numba, all you need is to import and add one decorator around the function (`njit`).

In [302]:
import numpy as np
from numba import njit

@njit
def sum_of_squares_numba(arr):
    s = 0.0
    for x in arr:
        s += x * x
    return s

arr_np = np.arange(1, 1_000_001, dtype=np.float64)

# first call triggers compilation — don't include in timing
_ = sum_of_squares_numba(arr_np[:10])

t0 = time.perf_counter()
result = sum_of_squares_numba(arr_np)
nb_sum_time = time.perf_counter() - t0

print(f"result = {result:.0f}")
print(f"time   = {nb_sum_time * 1000:.3f} ms")

result = 333333833333127552
time   = 0.542 ms


### Sum of Squares: Side by Side

All variants- Python, C at three optimization levels, and Numba (median of 5 runs).

In [303]:
import subprocess, os
import plotly.graph_objects as go

RUNS = 5

def bench_py(func, *args):
    times = []
    for _ in range(RUNS):
        t0 = time.perf_counter()
        func(*args)
        times.append((time.perf_counter() - t0) * 1000)
    return np.median(times), np.std(times)

def bench_c(binary):
    times = []
    for _ in range(RUNS):
        out = subprocess.run([binary], capture_output=True, text=True).stdout
        times.append(float(out.strip().split("time   = ")[1].split(" ms")[0]))
    return np.median(times), np.std(times)

def fmt_t(ms):
    if ms >= 1:    return f"{ms:.1f} ms"
    if ms >= 0.01: return f"{ms*1000:.0f} \u00b5s"
    return f"{ms*1000:.1f} \u00b5s"

BG       = "#fafaf8"
FONT_CLR = "#4a4540"
GRID_CLR = "#ece8e3"
FONT     = "Inter, system-ui, sans-serif"
MONO     = "JetBrains Mono, Consolas, monospace"
COLORS   = ["#e8a87c", "#b8e0de", "#85cdca", "#5ab8b4", "#c3aed6"]
cc_name  = os.path.basename(CC)

# ── benchmark sum of squares ─────────────────────────────────────────────────
names = ["Python", f"{cc_name} -O1", f"{cc_name} -O2", f"{cc_name} -O3", "Numba"]
results = [
    bench_py(sum_of_squares, arr),
    bench_c("./sum_sq_O1"),
    bench_c("./sum_sq_O2"),
    bench_c("./sum_sq_O3"),
    bench_py(sum_of_squares_numba, arr_np),
]
vals = [r[0] for r in results]
stds = [r[1] for r in results]

# reverse so fastest is at top
n_rev = names[::-1]
v_rev = vals[::-1]
s_rev = stds[::-1]
c_rev = COLORS[::-1]

fig = go.Figure(go.Bar(
    y=n_rev, x=v_rev, orientation="h",
    marker_color=c_rev,
    error_x=dict(type="data", array=s_rev, color="#999", thickness=1.5),
    text=[fmt_t(v) for v in v_rev],
    textposition="outside",
    textfont=dict(family=MONO, size=12, color=FONT_CLR),
    cliponaxis=False,
))

fig.update_layout(
    title=dict(text="Sum of Squares: Python vs C vs Numba",
               font=dict(size=15, color=FONT_CLR, family=FONT)),
    xaxis=dict(title=dict(text="time (ms, log scale)", font=dict(size=11)),
               type="log", gridcolor=GRID_CLR,
               tickfont=dict(family=MONO, size=11),
               tickformat=".3~g", ticksuffix=" ms"),
    yaxis=dict(tickfont=dict(family=FONT, size=12, color=FONT_CLR)),
    template="plotly_white", paper_bgcolor=BG, plot_bgcolor=BG,
    height=300, margin=dict(t=50, b=40, l=100, r=80),
)
fig.show()

#### Under the hood, Numba uses **LLVM** — the same compiler backend as Clang/C++ — to generate optimized machine code the first time you call the function.

---

## Conway's Game of Life
https://en.wikipedia.org/wiki/Conway%27s_Game_of_Life

Does the pattern hold for more complex examples ?
We'll implement and test the same on Conway's Game of life. It's grid based simulation where each cell lives or dies based on its 8 neighbors, heavy nested loops over a grid.

Given a grid with alive and dead cells, for every step, state updates using these rules-
  1. Underpopulation: alive cell with < 2 neighbors dies (lonely)
  2. Survival: alive cell with 2-3 neighbors lives
  3. Overpopulation: alive cell with > 3 neighbors dies (crowded)
  4. Reproduction: dead cell with exactly 3 neighbors becomes alive


### Python

In [311]:
def life_python(grid, steps):
    N, M = grid.shape
    for _ in range(steps):
        new = np.zeros_like(grid)
        for i in range(N):
            for j in range(M):
                nb = 0
                for di in (-1, 0, 1):
                    for dj in (-1, 0, 1):
                        if di == 0 and dj == 0:
                            continue
                        nb += grid[(i + di) % N, (j + dj) % M]
                if grid[i, j]:
                    new[i, j] = 1 if 2 <= nb <= 3 else 0
                else:
                    new[i, j] = 1 if nb == 3 else 0
        grid = new
    return grid

N_LIFE, STEPS = 64, 100
np.random.seed(42)
grid0 = np.random.randint(0, 2, (N_LIFE, N_LIFE), dtype=np.int8)
alive0 = int(grid0.sum())

t0 = time.perf_counter()
result = life_python(grid0, STEPS)
py_life_time = time.perf_counter() - t0

print(f"grid  = {N_LIFE}x{N_LIFE}, steps = {STEPS}")
print(f"alive: {alive0} -> {int(result.sum())}")
print(f"time  = {py_life_time * 1000:.1f} ms")

grid  = 64x64, steps = 100
alive: 2052 -> 303
time  = 356.9 ms


### C version

In [312]:
%%writefile life.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define N 64
#define STEPS 100

int main(void) {
    signed char *grid = malloc(N * N);
    signed char *next = malloc(N * N);

    srand(42);
    for (int i = 0; i < N * N; i++) grid[i] = rand() % 2;

    struct timespec t0, t1;
    clock_gettime(CLOCK_MONOTONIC, &t0);

    for (int s = 0; s < STEPS; s++) {
        for (int i = 0; i < N; i++)
            for (int j = 0; j < N; j++) {
                int nb = 0;
                for (int di = -1; di <= 1; di++)
                    for (int dj = -1; dj <= 1; dj++) {
                        if (di == 0 && dj == 0) continue;
                        nb += grid[((i+di+N)%N)*N + (j+dj+N)%N];
                    }
                next[i*N+j] = grid[i*N+j]
                    ? (nb == 2 || nb == 3)
                    : (nb == 3);
            }
        signed char *tmp = grid; grid = next; next = tmp;
    }

    clock_gettime(CLOCK_MONOTONIC, &t1);

    int alive = 0;
    for (int i = 0; i < N * N; i++) alive += grid[i];

    double elapsed = (t1.tv_sec - t0.tv_sec)
                   + (t1.tv_nsec - t0.tv_nsec) / 1e9;
    printf("alive = %d\n", alive);
    printf("time   = %.3f ms\n", elapsed * 1000);

    free(grid); free(next);
    return 0;
}

Overwriting life.c


In [313]:
!{CC} -O1 -o life_O1 life.c && echo "--- {CC} -O1 ---" && ./life_O1
!{CC} -O2 -o life_O2 life.c && echo "--- {CC} -O2 ---" && ./life_O2
!{CC} -O3 -o life_O3 life.c && echo "--- {CC} -O3 ---" && ./life_O3

--- /usr/bin/gcc -O1 ---
alive = 507
time   = 4.242 ms
--- /usr/bin/gcc -O2 ---
alive = 507
time   = 0.576 ms
--- /usr/bin/gcc -O3 ---
alive = 507
time   = 0.575 ms


### Numba version

Same Python function — add `@njit` and use a NumPy array.

In [314]:
@njit
def life_numba(grid, steps):
    N, M = grid.shape
    for _ in range(steps):
        new = np.zeros_like(grid)
        for i in range(N):
            for j in range(M):
                nb = 0
                for di in (-1, 0, 1):
                    for dj in (-1, 0, 1):
                        if di == 0 and dj == 0:
                            continue
                        nb += grid[(i + di) % N, (j + dj) % M]
                if grid[i, j]:
                    new[i, j] = 1 if 2 <= nb <= 3 else 0
                else:
                    new[i, j] = 1 if nb == 3 else 0
        grid = new
    return grid

# compile
_ = life_numba(np.random.randint(0, 2, (8, 8), dtype=np.int8), 1)

t0 = time.perf_counter()
result = life_numba(grid0, STEPS)
nb_life_time = time.perf_counter() - t0

print(f"alive: {alive0} -> {int(result.sum())}")
print(f"time  = {nb_life_time * 1000:.3f} ms")

alive: 2052 -> 303
time  = 2.134 ms


### Side by Side steps - Same Wall Time

Each frame: Python computes 1 step. Numba gets the same wall-time budget.
Watch the step counters diverge.

In [315]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

CMAP = mcolors.ListedColormap(['#fafaf8', '#2A9D8F'])
FRAMES = 50

np.random.seed(42)
grid_init = np.random.randint(0, 2, (N_LIFE, N_LIFE), dtype=np.int8)

# ── pre-compute: both actually execute, we record the grids ─────────────────
py_grids = [grid_init.copy()]
nb_grids = [grid_init.copy()]
py_ms    = [0.0]
nb_ms    = [0.0]
py_counts = [0]
nb_counts = [0]

py_g = grid_init.copy()
nb_g = grid_init.copy()
nb_total = 0

for frame in range(FRAMES):
    # python: 1 step, timed
    t0 = time.perf_counter()
    py_g = life_python(py_g, 1)
    py_dt = time.perf_counter() - t0

    # numba: as many steps as fit in the same wall time
    nb_frame = 0
    t0 = time.perf_counter()
    while time.perf_counter() - t0 < py_dt:
        nb_g = life_numba(nb_g, 1)
        nb_total += 1
        nb_frame += 1

    py_grids.append(py_g.copy())
    nb_grids.append(nb_g.copy())
    py_ms.append(py_dt * 1000)
    nb_ms.append(py_dt / nb_frame * 1000 if nb_frame else 0)
    py_counts.append(frame + 1)
    nb_counts.append(nb_total)

print(f"Python {FRAMES} steps, Numba {nb_total} steps ({nb_total // FRAMES}x)")

# ── animate (500ms per frame — slow enough to see Numba clearing the grid) ──
fig, (ax_py, ax_nb) = plt.subplots(1, 2, figsize=(11, 5))
plt.close(fig)

im_py = ax_py.imshow(grid_init, cmap=CMAP, vmin=0, vmax=1, interpolation='nearest')
im_nb = ax_nb.imshow(grid_init, cmap=CMAP, vmin=0, vmax=1, interpolation='nearest')
ax_py.set_xticks([]); ax_py.set_yticks([])
ax_nb.set_xticks([]); ax_nb.set_yticks([])

def update(i):
    im_py.set_data(py_grids[i])
    ax_py.set_title(f'Python  —  step {py_counts[i]}', fontsize=13, fontweight='bold')
    ax_py.set_xlabel(f'{py_ms[i]:.1f} ms / step', fontsize=10, color='#999')

    im_nb.set_data(nb_grids[i])
    ax_nb.set_title(f'Numba  —  step {nb_counts[i]}', fontsize=13, fontweight='bold')
    ax_nb.set_xlabel(f'{nb_ms[i]:.2f} ms / step', fontsize=10, color='#999')

    fig.suptitle(
        f'Same wall time  |  Python: {py_counts[i]}  vs  Numba: {nb_counts[i]} steps',
        fontsize=12, color='#4a4540')
    return [im_py, im_nb]

anim = FuncAnimation(fig, update, frames=FRAMES + 1, interval=500, blit=False)
HTML(anim.to_jshtml())

Python 50 steps, Numba 8254 steps (165x)


### Plot: Game of life

Same comparison 64x64 grid, 100 steps (median of 5 runs).

In [316]:
# ── benchmark game of life ───────────────────────────────────────────────────
names = ["Python", f"{cc_name} -O1", f"{cc_name} -O2", f"{cc_name} -O3", "Numba"]
results = [
    bench_py(life_python, grid0, STEPS),
    bench_c("./life_O1"),
    bench_c("./life_O2"),
    bench_c("./life_O3"),
    bench_py(life_numba, grid0, STEPS),
]
vals = [r[0] for r in results]
stds = [r[1] for r in results]

n_rev = names[::-1]
v_rev = vals[::-1]
s_rev = stds[::-1]
c_rev = COLORS[::-1]

fig = go.Figure(go.Bar(
    y=n_rev, x=v_rev, orientation="h",
    marker_color=c_rev,
    error_x=dict(type="data", array=s_rev, color="#999", thickness=1.5),
    text=[fmt_t(v) for v in v_rev],
    textposition="outside",
    textfont=dict(family=MONO, size=12, color=FONT_CLR),
    cliponaxis=False,
))

fig.update_layout(
    title=dict(text="Game of Life: Python vs C vs Numba",
               font=dict(size=15, color=FONT_CLR, family=FONT)),
    xaxis=dict(title=dict(text="time (ms, log scale)", font=dict(size=11)),
               type="log", gridcolor=GRID_CLR,
               tickfont=dict(family=MONO, size=11),
               tickformat=".3~g", ticksuffix=" ms"),
    yaxis=dict(tickfont=dict(family=FONT, size=12, color=FONT_CLR)),
    template="plotly_white", paper_bgcolor=BG, plot_bgcolor=BG,
    height=300, margin=dict(t=50, b=40, l=100, r=80),
)
fig.show()